# Annotation test run (validation set)

Runs a small LLM annotation test against the human validation gold: five
validation dialogues, all prompt templates, one model. Logic lives in
`extension/scripts/` (`prompt_loader`, `extraction`, `scoring`). Every call
caches per dialogue to `extension/artifacts/extraction_cache/{split}/{model}/{prompt}/{id}.json`
and re-runs skip valid entries. No output-token cap is set; models use their
provider defaults, and realised cost is measured from OpenRouter usage
accounting per call. This run explicitly requests maximum reasoning.

**Before running:** `export OPENROUTER_API_KEY=...` in the launching terminal.


In [1]:
import os, sys
from pathlib import Path
_here = Path.cwd()
for _c in [_here, *_here.parents]:
    if (_c / "extension" / "artifacts").exists():
        os.chdir(_c); break
sys.path.insert(0, str(Path.cwd()))
print("cwd:", os.getcwd(), "| key set:", bool(os.environ.get("OPENROUTER_API_KEY")))


cwd: /Users/tandon.utsav2/Desktop/Experiment_1 | key set: True


In [2]:
from extension.scripts.load_annotation_data import load_dataset
from extension.scripts import prompt_loader, extraction, scoring

gold = load_dataset("extension/artifacts/annotation_dev_and_val_sets/validation_set.csv")
# the validation gold was carved from the MathDial train split and keeps its
# train indices, so its cache lives under the train split: when the full
# train set is annotated later, these 78 dialogues are already done and skip
DIALOGUES = extraction.dialogues_from(gold, split="train")
UNITS = scoring.units_by_dialogue(gold)
print(f"{len(DIALOGUES)} dialogues, {len(gold)} units")


78 dialogues, 544 units


In [3]:
TEST_MODEL = 'moonshotai/kimi-k3'         # model slug sent to OpenRouter
TEST_REASONING_EFFORT = 'max'              # explicit maximum reasoning
TEST_PROMPTS = ['P1_full_codebook']       # any subset of prompt stems, e.g.
                                          # ['P1_full_codebook','P2_condensed_codebook',
                                          #  'P3_minimal','P4_condensed_staged']
N_TEST_DIALOGUES = 78
MAX_WORKERS = 1            # parallel in-flight requests; 4-6 is polite to a pinned provider
TEST_PROVIDER = 'modal/mxfp4'   # fastest OpenRouter host for Kimi K3; None for free
                          # routing (set None when testing other models unless
                          # you have checked they are served by the named host)


In [4]:
TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}; available: {prompt_loader.list_prompts()}"
print(f"test model: {TEST_MODEL} | reasoning: {TEST_REASONING_EFFORT} on dialogues "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x prompts {TEST_PROMPTS}\n")

import json as _json
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f"  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):")
    extraction.generate_annotations(
        prompt, TEST_MODEL, TEST_DIALOGUES, provider=TEST_PROVIDER,
        max_workers=MAX_WORKERS, reasoning_effort=TEST_REASONING_EFFORT,
    )
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(TEST_MODEL, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {'ok' if rec['valid'] else 'invalid':8s} "
              f"cost ${rec['cost_usd']:.4f}  latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, TEST_MODEL, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0,
                             split='train')
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['weighted_f1_P']:.3f} "
          f"| alpha {s['alpha']:.3f} | ${s['usd_per_dialogue']:.4f}/dialogue")
    print("     family F1(P): " + " | ".join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ) + "\n")

import pandas as pd
summary_cols = ['prompt', 'valid_rate', 'macro_f1_P', 'micro_f1_P',
                'weighted_f1_P', 'alpha',
                *family_f1_cols, 'usd_per_dialogue', 'latency_s']
summary = pd.DataFrame(test_rows)[summary_cols].round(3)
print(summary.to_string(index=False))


test model: moonshotai/kimi-k3 | reasoning: max on dialogues [1, 21, 35, 79, 143, 178, 255, 270, 275, 289, 300, 306, 323, 344, 351, 356, 380, 434, 448, 494, 532, 554, 589, 617, 635, 656, 695, 736, 758, 779, 818, 822, 842, 862, 947, 958, 966, 980, 992, 1026, 1051, 1063, 1071, 1084, 1089, 1107, 1119, 1217, 1221, 1300, 1349, 1421, 1452, 1490, 1519, 1540, 1553, 1555, 1557, 1571, 1615, 1658, 1668, 1717, 1719, 1776, 1780, 1870, 1890, 1916, 1941, 2018, 2164, 2192, 2196, 2202, 2206, 2222] x prompts ['P1_full_codebook']

  P1_full_codebook (78 dialogues, 1 workers):
  1: cached
  21: cached
  35: cached
  79: cached
  143: cached
  178: cached
  255: cached
  270: cached
  275: cached
  289: cached
  300: cached
  306: cached
  323: cached
  344: cached
  351: cached
  356: cached
  380: cached
  434: cached
  448: cached
  494: cached
  532: cached
  554: cached
  589: cached
  617: cached
  635: cached
  656: cached
  695: cached
  736: cached
  758: cached
  779: cached
  818: cached
  822: 

### Notes

Prompt files live in `extension/artifacts/annotation_prompts/`; dropping a
new `P*.md` there adds it to the run automatically. Every attempt is cached
with its raw output, full reasoning trace, validation errors, and usage,
so misreadings of the codebook can be diagnosed from the trace, and are
retried on the next execution. Explicit-max records use the existing
`moonshotai__kimi-k3` cache folder.
